# 🏗️ Notebook 4: Cluster and Replication

A single Redis server is fast, but it's a **single point of failure**. In production, Redis runs with **replicas** for redundancy, **Sentinel** for automatic failover, and optionally **Cluster mode** for horizontal scaling. In this notebook, we'll work hands-on with a master-replica setup and Redis Sentinel.

## Learning Objectives
- Understand master-replica replication and verify it works
- Learn how Redis Sentinel provides automatic failover
- Observe a simulated failover in action
- Understand Redis Cluster concepts: hash slots, sharding, MOVED redirects
- Learn strategies to mitigate the hot key problem

## 🛠️ Setup

This notebook uses the **full Docker setup** including the master, replicas, and sentinels.

```bash
cd 03-technologies/databases/redis
docker compose up -d
```

Verify all services are running:
```bash
docker compose ps
```

### Architecture

```
┌────────────┐    ┌────────────┐    ┌────────────┐
│   Redis     │───▶│  Replica 1 │    │  Replica 2 │
│   Master    │    │   :6381    │    │   :6382    │
│   :6380     │───▶└────────────┘    └────────────┘
└──────┬─────┘              ▲               ▲
       │                    │               │
┌──────┴────────────────────┴───────────────┴──┐
│          Sentinel Cluster (quorum=2)          │
│  Sentinel 1 :26379  │  2 :26380  │  3 :26381 │
└───────────────────────────────────────────────┘
```

### Visualization
- **RedisInsight**: http://localhost:5540
  - Connect to master at `localhost:6380`
  - Connect to replicas at `localhost:6381` and `localhost:6382`

### Kernel Selection
Select the `.venv` kernel in VS Code's kernel picker (top-right).
If it doesn't appear, reload: `Cmd+Shift+P` → "Reload Window".

In [1]:
import redis
import time
import json

# Three Redis nodes are running on these ports. Sentinel may have promoted any
# one of them to master after a previous failover demo, so we DON'T hardcode
# which port is master. We dynamically detect the role of each node.
NODE_PORTS = [6380, 6381, 6382]

# Connect to sentinels (used in later sections)
sentinel_hosts = [
    ("localhost", 26379),
    ("localhost", 26380),
    ("localhost", 26381),
]

def _role(port):
    try:
        c = redis.Redis(host="localhost", port=port, decode_responses=True, socket_timeout=2)
        return port, c, c.info("replication")["role"]
    except Exception as e:
        return port, None, f"unreachable: {e}"

def _port(client):
    return client.connection_pool.connection_kwargs["port"]

# Discover the current master and the two replicas.
nodes = [_role(p) for p in NODE_PORTS]
master = next((c for _, c, role in nodes if role == "master"), None)
replicas = [c for _, c, role in nodes if role == "slave"]

if master is None or len(replicas) < 2:
    print("Topology is not master + 2 replicas. Try restarting the stack:")
    print("   docker compose restart redis-master redis-replica-1 redis-replica-2 sentinel-1 sentinel-2 sentinel-3")
    print("   ... then wait ~10 s and re-run this cell.")
    print()
    print("Current roles:")
    for port, _, role in nodes:
        print(f"   :{port}  -> {role}")
else:
    replica1, replica2 = replicas[0], replicas[1]
    print("Topology detected:")
    for port, _, role in nodes:
        print(f"   :{port}  -> {role}")
    print(f"Using master = :{_port(master)}, replica1 = :{_port(replica1)}, replica2 = :{_port(replica2)}")


Topology detected:
   :6380  -> master
   :6381  -> slave
   :6382  -> slave
Using master = :6380, replica1 = :6381, replica2 = :6382


## 1️⃣ Master-Replica Replication

Redis replication is **asynchronous** by default. The master accepts writes and sends them to replicas in the background. Replicas are **read-only** — they cannot accept writes.

```
Write → Master → (async) → Replica 1
                         → Replica 2
Read ← Master or any Replica
```

Why replicas?
- **High availability**: if master fails, a replica can be promoted
- **Read scaling**: distribute read load across replicas
- **Data safety**: copies of your data on multiple machines

In [2]:
# Write to the master
master.set("message", "Hello from the master!")
print(f"✍️  Written to master: {master.get('message')}")

# Give replication a moment to propagate
time.sleep(0.5)

# Read from replicas — they should have the data!
print(f"📖 Read from replica 1: {replica1.get('message')}")
print(f"📖 Read from replica 2: {replica2.get('message')}")

print("\n💡 Data written to master is automatically replicated to all replicas!")

✍️  Written to master: Hello from the master!


📖 Read from replica 1: Hello from the master!
📖 Read from replica 2: Hello from the master!

💡 Data written to master is automatically replicated to all replicas!


In [3]:
# Replicas are READ-ONLY — writing to them fails
try:
    replica1.set("sneaky_write", "This should fail!")
    print("This line should never execute")
except redis.exceptions.ReadOnlyError as e:
    print(f"❌ Cannot write to replica: {e}")
    print("💡 Replicas are read-only by design — all writes must go through the master.")

❌ Cannot write to replica: You can't write against a read only replica.
💡 Replicas are read-only by design — all writes must go through the master.


In [4]:
# Check replication status
info = master.info("replication")
print("📊 Master Replication Info:")
print(f"   Role: {info['role']}")
print(f"   Connected replicas: {info['connected_slaves']}")

for i in range(info['connected_slaves']):
    slave_info = info[f'slave{i}']
    print(f"   Replica {i}: {slave_info}")

# Check from replica's perspective
info_r = replica1.info("replication")
print(f"\n📊 Replica 1 Info:")
print(f"   Role: {info_r['role']}")
print(f"   Master host: {info_r['master_host']}")
print(f"   Master port: {info_r['master_port']}")
print(f"   Master link status: {info_r['master_link_status']}")

📊 Master Replication Info:
   Role: master
   Connected replicas: 2
   Replica 0: {'ip': '172.29.0.5', 'port': 6379, 'state': 'online', 'offset': 5874, 'lag': 0}
   Replica 1: {'ip': '172.29.0.6', 'port': 6379, 'state': 'online', 'offset': 5874, 'lag': 0}

📊 Replica 1 Info:
   Role: slave
   Master host: redis-master
   Master port: 6379
   Master link status: up


### Replication Lag

Because replication is asynchronous, there's a brief window where replicas may not have the latest data. Let's measure it.

In [5]:
# Write many keys rapidly and check if replicas keep up
print("⏱️  Measuring replication lag...\n")

total_writes = 100
lag_detected = 0

for i in range(total_writes):
    key = f"lag_test:{i}"
    master.set(key, f"value_{i}")

    # Immediately check replica (no sleep!)
    val = replica1.get(key)
    if val is None:
        lag_detected += 1

print(f"Wrote {total_writes} keys to master")
print(f"Immediate reads from replica: {lag_detected} had lag (data not yet replicated)")
print(f"Lag rate: {lag_detected/total_writes*100:.1f}%")

# After a brief pause, all data should be there
time.sleep(0.5)
final_check = sum(1 for i in range(total_writes) if replica1.get(f"lag_test:{i}") is not None)
print(f"\nAfter 500ms: {final_check}/{total_writes} keys visible on replica")

# Clean up
for i in range(total_writes):
    master.delete(f"lag_test:{i}")
master.delete("message")

print("\n💡 Replication lag is usually sub-millisecond, but it exists!")
print("   This is why you should read from master for critical/fresh data.")

⏱️  Measuring replication lag...

Wrote 100 keys to master
Immediate reads from replica: 0 had lag (data not yet replicated)
Lag rate: 0.0%



After 500ms: 100/100 keys visible on replica

💡 Replication lag is usually sub-millisecond, but it exists!
   This is why you should read from master for critical/fresh data.


## 2️⃣ Redis Sentinel

Sentinel is Redis' **high-availability** solution. It monitors the master and replicas, and automatically promotes a replica to master if the master fails.

```
Normal operation:          After master fails:
Sentinel monitors          Sentinel detects failure
     ↓                          ↓
Master ← Replicas         Replica 1 promoted to Master!
                                ↓
                           Replica 2 replicates from new master
```

Our setup has 3 Sentinels with a **quorum of 2** — at least 2 Sentinels must agree that the master is down before failover begins.

In [6]:
# Sentinels live INSIDE the Docker network. They identify the master by its
# Docker hostname ("redis-master") and the replicas by their container IPs
# (e.g. 172.29.0.6) — neither is reachable from the host machine.
#
# We build a small lookup that translates whatever Sentinel reports
# (hostname OR container IP) to the host-mapped port (6380/6381/6382).

import subprocess
from redis.sentinel import Sentinel

sentinel = Sentinel(sentinel_hosts, socket_timeout=5)

CONTAINER_TO_HOST_PORT = {
    "redis-master":    6380,
    "redis-replica-1": 6381,
    "redis-replica-2": 6382,
}

def _docker_ip(name):
    out = subprocess.run(
        ["docker", "inspect", name,
         "--format", "{{range .NetworkSettings.Networks}}{{.IPAddress}}{{end}}"],
        capture_output=True, text=True,
    )
    return out.stdout.strip()

# Build a fresh IP -> container-name map at notebook start
IP_TO_CONTAINER = {_docker_ip(name): name for name in CONTAINER_TO_HOST_PORT}
print("Container IP -> name map:")
for ip, name in IP_TO_CONTAINER.items():
    print(f"  {ip} -> {name}")

def host_addr(host_or_ip, _port):
    """Translate Sentinel's reported address to a host-reachable one.

    Sentinel returns hostnames (e.g. 'redis-master') or container IPs
    depending on configuration. We always come back to ('localhost', host_port).
    """
    name = host_or_ip if host_or_ip in CONTAINER_TO_HOST_PORT else IP_TO_CONTAINER.get(host_or_ip)
    if name is None:
        return host_or_ip, int(_port)
    return "localhost", CONTAINER_TO_HOST_PORT[name]

def connect_to_master():
    """Ask Sentinel for the current master, then connect via host ports."""
    host, port = sentinel.discover_master("mymaster")
    real_host, real_port = host_addr(host, port)
    return (
        redis.Redis(host=real_host, port=real_port, decode_responses=True, socket_timeout=2),
        (host, int(port)),
    )

def connect_to_replica():
    """Ask Sentinel for a replica and return a host-reachable client."""
    replicas = sentinel.discover_slaves("mymaster")
    if not replicas:
        return None, None
    host, port = replicas[0]
    real_host, real_port = host_addr(host, port)
    return (
        redis.Redis(host=real_host, port=real_port, decode_responses=True, socket_timeout=2),
        (host, int(port)),
    )

# Ask Sentinel: who is the current master?
m_host, m_port = sentinel.discover_master("mymaster")
print(f"\nSentinel reports master at: {m_host}:{m_port} (docker-internal)")
real = host_addr(m_host, m_port)
print(f"   Host-reachable address:  {real[0]}:{real[1]}")

# Ask Sentinel: who are the replicas?
print("\nKnown replicas:")
for host, port in sentinel.discover_slaves("mymaster"):
    real = host_addr(host, port)
    print(f"   {host}:{port}  (host: {real[0]}:{real[1]})")

# Write through the helper - works no matter which node is master
m, _ = connect_to_master()
m.set("via_sentinel", "Written through Sentinel-aware helper!")
print(f"\nWritten via helper: {m.get('via_sentinel')}")

rep, addr = connect_to_replica()
if rep:
    time.sleep(0.3)
    print(f"Read from replica {addr[0]}:{addr[1]}: {rep.get('via_sentinel')}")

print("\n[TIP] In real production deployments (no docker port mapping),")
print("      `sentinel.master_for(...)` works directly because the addresses")
print("      Sentinel returns are reachable from your application servers.")


Container IP -> name map:
  172.29.0.3 -> redis-master
  172.29.0.6 -> redis-replica-1
  172.29.0.5 -> redis-replica-2

Sentinel reports master at: redis-master:6379 (docker-internal)
   Host-reachable address:  localhost:6380

Known replicas:
   172.29.0.6:6379  (host: localhost:6381)
   172.29.0.5:6379  (host: localhost:6382)

Written via helper: Written through Sentinel-aware helper!


Read from replica 172.29.0.6:6379: Written through Sentinel-aware helper!

[TIP] In real production deployments (no docker port mapping),
      `sentinel.master_for(...)` works directly because the addresses
      Sentinel returns are reachable from your application servers.


In [7]:
# Check Sentinel health and configuration
import socket

for i, (host, port) in enumerate(sentinel_hosts, 1):
    try:
        s = redis.Redis(host=host, port=port, decode_responses=True)

        # SENTINEL masters — list monitored masters
        masters = s.sentinel_masters()
        for name, info in masters.items():
            print(f"Sentinel {i} (:{port}) → Master '{name}':")
            print(f"  Address: {info['ip']}:{info['port']}")
            print(f"  Status:  {info['flags']}")
            print(f"  Replicas: {info['num-slaves']}")
            print(f"  Sentinels: {info['num-other-sentinels'] + 1}")
            print(f"  Quorum: {info['quorum']}")
            print()
    except Exception as e:
        print(f"Sentinel {i} (:{port}): ❌ {e}\n")

Sentinel 1 (:26379) → Master 'mymaster':
  Address: redis-master:6379
  Status:  master
  Replicas: 2
  Sentinels: 3
  Quorum: 2

Sentinel 2 (:26380) → Master 'mymaster':
  Address: redis-master:6379
  Status:  master
  Replicas: 2
  Sentinels: 3
  Quorum: 2

Sentinel 3 (:26381) → Master 'mymaster':
  Address: redis-master:6379
  Status:  master
  Replicas: 2
  Sentinels: 3
  Quorum: 2



## 3️⃣ Simulating Failover

Let's watch what happens when the master goes down! We'll pause the master container and observe Sentinel promoting a replica.

⚠️ **This cell stops and restarts Docker containers.** The failover takes about 10-15 seconds.

In [8]:
# We already imported subprocess in the previous cell.

def run_cmd(cmd):
    """Run a shell command and return output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout.strip()

# Step 1: Current master (via helper)
print("Step 1: Current master")
master_conn, master_internal = connect_to_master()
# master_internal might be hostname OR IP -> resolve to a container name we can pause
container_name = (
    master_internal[0]
    if master_internal[0] in CONTAINER_TO_HOST_PORT
    else IP_TO_CONTAINER.get(master_internal[0], master_internal[0])
)
print(f"  Internal: {master_internal[0]}:{master_internal[1]}  (container: {container_name})")
master_conn.set("before_failover", "I was written before the crash!")
print("  Written: 'before_failover' = 'I was written before the crash!'")

# Step 2: Pause the master container
print(f"\nStep 2: Pausing the master container ({container_name})...")
run_cmd(f"docker pause {container_name}")
print("  Master paused. Sentinel will detect the failure...")

# Step 3: Wait for failover
print("\nStep 3: Waiting for failover (~10-20 seconds)...")
new_master_internal = master_internal
for i in range(30):
    time.sleep(1)
    try:
        addr = sentinel.discover_master("mymaster")
        if addr != master_internal:
            new_master_internal = addr
            real = host_addr(*addr)
            print(f"  Failover detected after ~{i+1} seconds!")
            print(f"  New master: {addr[0]}:{addr[1]} (host: {real[0]}:{real[1]})")
            break
    except Exception:
        pass
    if (i + 1) % 3 == 0:
        print(f"  ... waiting ({i+1}s)")
else:
    print("  Failover not detected within 30s")

# Step 4: Verify data survived
time.sleep(2)
try:
    new_master_conn, _ = connect_to_master()
    value = new_master_conn.get("before_failover")
    print(f"\nStep 4: Data check after failover")
    print(f"  'before_failover' = '{value}'")
    if value:
        print("  Data survived the failover!")
    new_master_conn.set("after_failover", "Written to the new master!")
    print("  Written to new master: 'after_failover'")
except Exception as e:
    print(f"  Error connecting to new master: {e}")

# Step 5: Restore old master
print(f"\nStep 5: Unpausing old master container ({container_name})...")
run_cmd(f"docker unpause {container_name}")
time.sleep(3)
print("  Old master unpaused - it will rejoin as a replica")

time.sleep(2)
final_master = sentinel.discover_master("mymaster")
final_replicas = sentinel.discover_slaves("mymaster")
print("\nFinal topology:")
print(f"  Master: {final_master[0]}:{final_master[1]}")
for host, port in final_replicas:
    print(f"  Replica: {host}:{port}")

print("\n[TIP] Sentinel handled failover automatically.")
print("      The old master rejoined as a replica after being unpaused.")


Step 1: Current master
  Internal: redis-master:6379  (container: redis-master)
  Written: 'before_failover' = 'I was written before the crash!'

Step 2: Pausing the master container (redis-master)...
  Master paused. Sentinel will detect the failure...

Step 3: Waiting for failover (~10-20 seconds)...


  ... waiting (3s)


  Failover detected after ~6 seconds!
  New master: 172.29.0.5:6379 (host: localhost:6382)



Step 4: Data check after failover
  'before_failover' = 'I was written before the crash!'
  Data survived the failover!
  Written to new master: 'after_failover'

Step 5: Unpausing old master container (redis-master)...


  Old master unpaused - it will rejoin as a replica



Final topology:
  Master: 172.29.0.5:6379
  Replica: 172.29.0.6:6379
  Replica: redis-master:6379

[TIP] Sentinel handled failover automatically.
      The old master rejoined as a replica after being unpaused.


## 4️⃣ Redis Cluster Concepts

While our lab uses Sentinel (for HA), production systems often use **Redis Cluster** for both HA and horizontal scaling. Here's how it works conceptually.

### Hash Slots

Redis Cluster divides the key space into **16,384 hash slots**. Each key is assigned to a slot using `CRC16(key) mod 16384`.

```
Node A: slots 0-5460       Node B: slots 5461-10922    Node C: slots 10923-16383
┌──────────────────┐       ┌──────────────────┐       ┌──────────────────┐
│  user:1 → slot 3 │       │ user:2 → slot 7000│       │ user:3 → slot 12000│
│  user:4 → slot 50│       │ user:5 → slot 8000│       │ user:6 → slot 15000│
└──────────────────┘       └──────────────────┘       └──────────────────┘
```

### How Clients Route Requests

1. Client computes `CRC16(key) mod 16384` to find the slot
2. Client checks its local **slot map** to find which node owns the slot
3. Client sends the command directly to that node
4. If the slot moved (during rebalancing), the node replies with `MOVED` and the client refreshes its map

In [9]:
# Let's simulate how Redis Cluster assigns keys to slots
# This uses the same CRC16 algorithm as Redis

import binascii

def redis_slot(key):
    """Calculate the Redis Cluster hash slot for a key."""
    # Redis uses CRC16 (CCITT variant) mod 16384
    # Handle hash tags: if key contains {...}, only the content in {} is hashed
    start = key.find('{')
    if start != -1:
        end = key.find('}', start + 1)
        if end != -1 and end != start + 1:
            key = key[start + 1:end]

    # CRC16 (simplified — real Redis uses CCITT)
    crc = binascii.crc_hqx(key.encode(), 0)
    return crc % 16384

# Show how different keys map to different slots
print("🎰 Hash Slot Assignment (16,384 total slots)")
print("=" * 55)
keys = ["user:1", "user:2", "user:3", "product:100", "order:500", "session:abc"]
for key in keys:
    slot = redis_slot(key)
    node = "Node A" if slot < 5461 else "Node B" if slot < 10923 else "Node C"
    print(f"  {key:20s} → slot {slot:5d} → {node}")

# Hash tags: force related keys to the same slot
print("\n🏷️  Hash Tags — force keys to the same slot:")
tagged_keys = ["{user:1}:profile", "{user:1}:orders", "{user:1}:sessions"]
for key in tagged_keys:
    slot = redis_slot(key)
    print(f"  {key:25s} → slot {slot:5d}")

print("\n💡 All {user:1}:* keys land on the same slot → same node!")
print("   This enables multi-key operations (MGET, transactions) in Cluster mode.")

🎰 Hash Slot Assignment (16,384 total slots)
  user:1               → slot 10778 → Node B
  user:2               → slot  6777 → Node B
  user:3               → slot  2648 → Node A
  product:100          → slot  9618 → Node B
  order:500            → slot 12746 → Node C
  session:abc          → slot 14788 → Node C

🏷️  Hash Tags — force keys to the same slot:
  {user:1}:profile          → slot 10778
  {user:1}:orders           → slot 10778
  {user:1}:sessions         → slot 10778

💡 All {user:1}:* keys land on the same slot → same node!
   This enables multi-key operations (MGET, transactions) in Cluster mode.


## 5️⃣ The Hot Key Problem

When one key gets dramatically more traffic than others, the node holding that key becomes a bottleneck. This is the **hot key problem**.

```
Normal:  Traffic evenly spread across nodes
         Node A: 33%  Node B: 33%  Node C: 33%

Hot key: One product goes viral
         Node A: 80%  Node B: 10%  Node C: 10%
         Node A is overwhelmed! 🔥
```

### Mitigation Strategies

In [10]:
r_standalone = redis.Redis(host="localhost", port=6379, decode_responses=True)
r_standalone.flushdb()

# Strategy 1: Client-side caching (in-process cache)
print("Strategy 1: Client-Side Cache")
print("-" * 40)

local_cache = {}
LOCAL_CACHE_TTL = 5  # seconds

def get_with_local_cache(key):
    """Check local (in-process) cache before hitting Redis."""
    now = time.time()

    if key in local_cache:
        value, expires_at = local_cache[key]
        if now < expires_at:
            return value, "LOCAL_HIT"
        else:
            del local_cache[key]

    # Local miss — fetch from Redis
    value = r_standalone.get(key)
    if value:
        local_cache[key] = (value, now + LOCAL_CACHE_TTL)
    return value, "REDIS_HIT"

r_standalone.set("viral_product", json.dumps({"name": "Hot Item", "price": 9.99}))

for i in range(5):
    value, source = get_with_local_cache("viral_product")
    print(f"  Request {i+1}: {source}")

print("💡 Only the first request hits Redis; subsequent ones use local memory.\n")

# Strategy 2: Key replication with random suffix
print("Strategy 2: Key Replication (read from random replica)")
print("-" * 40)

import random

NUM_REPLICAS = 5

def write_with_replicas(base_key, value):
    """Write the same data under multiple keys to spread reads."""
    for i in range(NUM_REPLICAS):
        r_standalone.set(f"{base_key}:replica:{i}", value)

def read_from_replica(base_key):
    """Read from a random replica key — spreads load across slots."""
    replica_num = random.randint(0, NUM_REPLICAS - 1)
    key = f"{base_key}:replica:{replica_num}"
    return r_standalone.get(key), key

# Write the viral product to 5 different keys
write_with_replicas("viral_product_v2", json.dumps({"name": "Hot Item", "price": 9.99}))

# Reads are randomly distributed
distribution = {}
for _ in range(100):
    value, key_used = read_from_replica("viral_product_v2")
    distribution[key_used] = distribution.get(key_used, 0) + 1

print("  Read distribution across replica keys (100 reads):")
for key, count in sorted(distribution.items()):
    bar = "█" * (count // 2)
    print(f"    {key}: {count} reads {bar}")

print("💡 Reads are spread across 5 keys → 5 different hash slots → up to 5 nodes!")

r_standalone.flushdb()

Strategy 1: Client-Side Cache
----------------------------------------
  Request 1: REDIS_HIT
  Request 2: LOCAL_HIT
  Request 3: LOCAL_HIT
  Request 4: LOCAL_HIT
  Request 5: LOCAL_HIT
💡 Only the first request hits Redis; subsequent ones use local memory.

Strategy 2: Key Replication (read from random replica)
----------------------------------------
  Read distribution across replica keys (100 reads):
    viral_product_v2:replica:0: 19 reads █████████
    viral_product_v2:replica:1: 19 reads █████████
    viral_product_v2:replica:2: 22 reads ███████████
    viral_product_v2:replica:3: 15 reads ███████
    viral_product_v2:replica:4: 25 reads ████████████
💡 Reads are spread across 5 keys → 5 different hash slots → up to 5 nodes!


True

## 📋 Infrastructure Comparison

| Feature | Single Node | Master + Replicas | Sentinel | Cluster |
|---------|------------|-------------------|----------|--------|
| **High Availability** | ❌ | Manual failover | ✅ Automatic | ✅ Automatic |
| **Read Scaling** | ❌ | ✅ Read from replicas | ✅ | ✅ |
| **Write Scaling** | ❌ | ❌ Single master | ❌ Single master | ✅ Multi-master |
| **Data Capacity** | Single node RAM | Single node RAM | Single node RAM | Sum of all nodes |
| **Multi-key Ops** | ✅ | ✅ | ✅ | ⚠️ Same slot only |
| **Complexity** | Low | Low | Medium | High |
| **Best For** | Dev/test | Read-heavy + HA | Most production | Large scale |

## 🧹 Cleanup

In [11]:
# Clean up all connections
for client in [master, replica1, replica2]:
    try:
        if client.info("replication")["role"] == "master":
            client.flushdb()
    except Exception:
        pass

r_standalone = redis.Redis(host="localhost", port=6379, decode_responses=True)
r_standalone.flushdb()

print("🧹 Cleaned up all keys")
print("\nTo stop all Docker services:")
print("  cd 03-technologies/databases/redis && docker compose down")

🧹 Cleaned up all keys

To stop all Docker services:
  cd deep-dives/redis && docker compose down


## 📚 Summary

### Key Takeaways

1. **Replication** is asynchronous — replicas may briefly lag behind the master
2. **Replicas are read-only** — all writes must go through the master
3. **Sentinel** monitors the master and automatically promotes a replica on failure
4. **Always connect through Sentinel** in production — it handles failover transparently
5. **Redis Cluster** uses 16,384 hash slots to distribute keys across nodes
6. **Hash tags** `{...}` force related keys to the same slot for multi-key operations
7. **Hot keys** can overwhelm a single node — mitigate with local caching or key replication
8. **Choose Sentinel** for most production setups; **choose Cluster** when you need write scaling or more RAM than one node provides

### 🎓 Course Complete!

Congratulations! You've completed the Redis Deep Dive. You now understand:
- ✅ Redis data structures and when to use each
- ✅ Pub/Sub for real-time messaging and Streams for durable events
- ✅ Caching patterns, distributed locks, rate limiters, and leaderboards
- ✅ Replication, Sentinel, Cluster, and hot key mitigation

These concepts appear constantly in system design interviews. The key insight is that Redis is incredibly versatile — mastering one technology deeply is more valuable than knowing many technologies superficially.